<div align="center">

# Day 05: Direct Preference Optimization and Evaluation

**Reinforcement Learning and Language Model Alignment (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-05) · [Lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html) · [Interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lab.html) · [PDF notes](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/Day05_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this notebook is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from itertools import product
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.4), 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 9})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


import time
print('ready')

## Python step 5: Objects, copies and the DPO loss

The reference policy must stay frozen while the policy changes, and the optimum and the loss of direct preference optimisation (DPO) follow from a few probabilities. This step shows both in plain Python and NumPy &#91;[5](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-5), [6](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-6)&#93;.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "time" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### An alias is not a copy

In [ ]:
import copy
class Policy:
    def __init__(self, weights):
        self.weights = weights

pol = Policy([1.0, 2.0])
alias = pol                     # a second name for the same object
frozen = copy.deepcopy(pol)     # an independent object with the same content
pol.weights[0] = 99.0
print("alias :", alias.weights)
print("frozen:", frozen.weights)

Assignment in Python never copies an object; it gives the object another name. A reference policy created by `ref = pol` would change with every update of the policy, and the Kullback-Leibler (KL) divergence between the two would always be zero. `copy.deepcopy`, or the `copy` method of the language model in the notebook, creates an independent object.

### The optimum with NumPy arrays

In [ ]:
import numpy as np
p_ref = np.array([0.4, 0.3, 0.2, 0.1])   # probabilities under the reference
r = np.array([0.0, 0.5, 1.0, 2.0])       # rewards of the four sentences
for beta in (1.0, 0.25):
    w = p_ref * np.exp(r / beta)
    p_opt = w / w.sum()                  # divided by Z: the optimum
    reward = (p_opt * r).sum()
    kl = (p_opt * np.log(p_opt / p_ref)).sum()
    print(f"beta {beta}: optimum {p_opt.round(3)}, reward {reward:.2f}, KL {kl:.2f}")

Arithmetic on NumPy arrays works element by element: `p_ref * np.exp(r / beta)` multiplies every reference probability by its own factor, and `w.sum()` is the number $Z$ of the lecture. The KL divergence is the mean of $\log \pi^*(y) - \log \pi_{\mathrm{ref}}(y)$ under the optimum, which for four sentences is a sum weighted by `p_opt`. With $\beta = 0.25$ almost all probability moves to the sentence with the highest reward: The expected reward rises from 1.04 to 1.95, and the KL divergence from 0.26 to 2.07.

### A function that returns a dictionary

In [ ]:
import pandas as pd

def sigmoid(u):
    return 1 / (1 + np.exp(-u))

def dpo_pair(lw, ll, lw_ref, ll_ref, beta=0.1):
    """Margin, DPO loss and gradient weight of one pair."""
    z = (lw - lw_ref) - (ll - ll_ref)
    loss = -np.log(sigmoid(beta * z))
    return {"margin": z, "loss": loss, "weight": sigmoid(-beta * z)}

pairs = {"ranked correctly": (-5.0, -15.0, -10.0, -10.0),
         "ranked wrongly": (-15.0, -5.0, -10.0, -10.0)}
rows = [dpo_pair(*logps) for logps in pairs.values()]
print(pd.DataFrame(rows, index=list(pairs)).round(3))

`dpo_pair` computes the margin $z$, the loss $-\log \sigma(\beta z)$ and the gradient weight $\sigma(-\beta z)$ of the lecture and returns them as a dictionary, so that every number keeps its name. In `dpo_pair(*logps)` the star unpacks the tuple into the four log-probabilities, and a list of dictionaries with the same keys becomes a data frame with one row per pair. In both pairs the reference gives each sentence the log-probability $-10$, so the margin depends only on the policy. The pair that the policy ranks wrongly has the larger loss and pulls almost three times as hard.

**Quick check.** What would go wrong if the reference policy of reinforcement learning from human feedback (RLHF) were created with `ref = pol`?

<details><summary>Answer</summary>

The reference would be the same object as the policy and change with it, so the KL penalty would always be zero and the leash of Day 4 would disappear.

</details>

### Exercises for Python step 5

**Exercise 5.1.** Store the first weight of `alias` after the change in `a0`.

In [ ]:
a0 = None   # your answer

check("Exercise 5.1", a0, 99.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
a0 = alias.weights[0]

check("Exercise 5.1", a0, 99.0)

**Exercise 5.2.** Store the first weight of `frozen` in `f0`.

In [ ]:
f0 = None   # your answer

check("Exercise 5.2", f0, 1.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
f0 = frozen.weights[0]

check("Exercise 5.2", f0, 1.0)

**Exercise 5.3.** A policy that still equals the reference gives the winner of a pair the log-probability `-9.0` and the loser `-12.0`, exactly as the reference does. Use `dpo_pair` to store the loss of this pair in `loss_ref`.

In [ ]:
loss_ref = None   # your answer

check("Exercise 5.3", loss_ref, np.log(2))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
loss_ref = dpo_pair(-9.0, -12.0, -9.0, -12.0)["loss"]

check("Exercise 5.3", loss_ref, np.log(2))

## 1. The world, the fine-tuned model, the pairs and the reward model of Day 4

*Lecture: [From the optimum to DPO](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#from-the-optimum-to-dpo)*

The notebook rebuilds them, so that it can run on its own. The last cell of the section defines RLHF as the function `rlhf`, which section 3 runs.

**From the formulas to the code.** `gold_reward` is the gold reward of Day 4, and `GRAMMAR` is the set of the 1512 sentences of the grammar. `VOCAB` holds the 38 tokens: 35 words and the three special tokens `<pad>`, `<bos>` and `<eos>` for padding and for the beginning and the end of a sentence. `seq_logprob` returns $\log \pi_\theta(y)$, the sum of the token log-probabilities with the end token included and the padding left out. `sft` is the model after supervised fine-tuning (SFT), and it later serves as the reference $\pi_{\mathrm{ref}}$. `Wp` and `Lp` hold the winners $y_w$ and the losers $y_l$ of the 1372 pairs, and `train_rm` fits the reward model $r_\phi$ to the first 85 percent of them. In `rlhf`, the line `reward = rm.score(seqs) - beta * kl` is the penalised reward $\tilde{r}(y)$ of the lecture, where `kl` holds the difference $\log \pi_\theta(y) - \log \pi_{\mathrm{ref}}(y)$ of every sampled sentence.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "loss_ref" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# The large TokenWorld: short English sentences built from four slots, and the gold reward.
SUBJ = ["the cat", "the dog", "a bird", "my friend", "the child", "a robot"]
VERB = ["sat", "ran", "slept", "sang", "waited", "smiled"]
PLACE = ["on the mat", "in the garden", "by the window", "near the door", "under the tree", "beside the lamp"]
EXTRA = ["quietly", "happily", "again", "for a while", "today", "at dawn"]

def make_corpus(n=4000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        parts = [rng.choice(SUBJ), rng.choice(VERB), rng.choice(PLACE)]
        if rng.random() < 0.45:
            parts.append(rng.choice(EXTRA))
        out.append(" ".join(parts))
    return out

CALM = {"quietly", "slept", "waited", "beside", "under", "garden", "window", "dawn"}
LOUD = {"sang", "ran", "happily", "again", "robot"}
FUNC = {"the", "a", "my", "in", "on", "by", "at", "near", "under", "beside", "for", "while"}

def gold_reward(text):
    """The reward that is wanted: a calm, well-formed, non-repetitive, reasonably short sentence."""
    w = text.split()
    if not w:
        return -2.0
    content = [x for x in w if x not in FUNC]
    r = 1.0 if w[0] in ("the", "a", "my") else -0.5          # a grammatical opening
    r += 0.45 * len(CALM.intersection(w)) - 0.45 * len(LOUD.intersection(w))
    r -= 0.10 * max(0, len(w) - 6)                           # a mild preference for brevity
    if len(set(content)) < len(content):
        r -= 0.8                                             # repeated content words
    return float(r)

corpus = make_corpus()
VOCAB = ["<pad>", "<bos>", "<eos>"] + sorted({w for s in corpus for w in s.split()})
STOI = {w: i for i, w in enumerate(VOCAB)}; PAD, BOS, EOS = 0, 1, 2; V = len(VOCAB); MAXLEN = 12

def encode(s):
    ids = [BOS] + [STOI[w] for w in s.split()][:MAXLEN - 2] + [EOS]
    return ids + [PAD] * (MAXLEN - len(ids))

def decode(row):
    out = []
    for i in row[1:]:
        if i == EOS or i == PAD:
            break
        out.append(VOCAB[i])
    return " ".join(out)

IDS = np.array([encode(s) for s in corpus])
GRAMMAR = {" ".join([s, v, p] + ([e] if e else [])) for s in SUBJ for v in VERB for p in PLACE for e in EXTRA + [None]}
print(f"{len(corpus)} sentences, vocabulary of {V} tokens; example: '{corpus[0]}'")
print(f"gold reward of the corpus: mean {np.mean([gold_reward(s) for s in corpus]):.3f}")

In [ ]:
class NeuralLM:
    """A neural language model: each next token is predicted from the previous K tokens and the position.

    Every training objective of Days 4 and 5 is a weighted sum of token log-probabilities, so one backward pass,
    `grad_weighted`, serves supervised fine-tuning, REINFORCE and DPO alike."""
    def __init__(self, K=3, d=16, H=128, seed=0):
        rng = np.random.default_rng(seed); self.K, self.d = K, d
        self.P = {"E": rng.normal(0, 0.3, (V, d)), "pos": rng.normal(0, 0.3, (MAXLEN, d)),
                  "W1": rng.normal(0, 1 / np.sqrt((K + 1) * d), ((K + 1) * d, H)), "b1": np.zeros(H),
                  "W2": rng.normal(0, 1 / np.sqrt(H), (H, V)), "b2": np.zeros(V)}
        self.m = {k: np.zeros_like(v) for k, v in self.P.items()}; self.v = {k: np.zeros_like(v) for k, v in self.P.items()}
        self.t = 0

    def copy(self):
        c = NeuralLM.__new__(NeuralLM); c.K, c.d = self.K, self.d
        c.P = {k: v.copy() for k, v in self.P.items()}
        c.m = {k: np.zeros_like(v) for k, v in self.P.items()}; c.v = {k: np.zeros_like(v) for k, v in self.P.items()}; c.t = 0
        return c

    def _context(self, ids):
        N, T = ids.shape
        ext = np.concatenate([np.full((N, self.K - 1), BOS), ids[:, :-1]], axis=1)
        return np.stack([ext[:, k:k + T - 1] for k in range(self.K)], axis=2)       # (N, T-1, K)

    def forward(self, ids):
        """Log-probabilities of every next token, shape (N, MAXLEN-1, V)."""
        N, T = ids.shape; ctx = self._context(ids); pos = np.broadcast_to(self.P["pos"][:T - 1], (N, T - 1, self.d))
        X = np.concatenate([self.P["E"][ctx].reshape(N, T - 1, -1), pos], axis=2)
        Hh = np.tanh(X @ self.P["W1"] + self.P["b1"]); logits = Hh @ self.P["W2"] + self.P["b2"]
        logits[..., PAD] = -1e9                                                         # padding is never generated
        lsm = logits - logits.max(-1, keepdims=True); lsm = lsm - np.log(np.exp(lsm).sum(-1, keepdims=True))
        self.cache = (ids, ctx, X, Hh, lsm); return lsm

    def token_logprobs(self, ids):
        lsm = self.forward(ids); y = ids[:, 1:]
        return np.take_along_axis(lsm, y[..., None], 2)[..., 0], (y != PAD).astype(float)

    def seq_logprob(self, ids):
        lp, mask = self.token_logprobs(ids); return (lp * mask).sum(1)

    def grad_weighted(self, ids, w):
        """Gradient of  -sum_n w_n * log pi(sequence n)  with respect to every parameter."""
        lsm = self.forward(ids); ids_, ctx, X, Hh, _ = self.cache; N, T1 = ctx.shape[:2]; y = ids[:, 1:]
        Wt = (y != PAD) * np.asarray(w, float)[:, None]
        G = np.exp(lsm) * Wt[..., None]; np.put_along_axis(G, y[..., None], np.take_along_axis(G, y[..., None], 2) - Wt[..., None], 2)
        g = {"W2": Hh.reshape(-1, Hh.shape[-1]).T @ G.reshape(-1, V), "b2": G.sum((0, 1))}
        dpre = (G @ self.P["W2"].T) * (1 - Hh ** 2)
        g["W1"] = X.reshape(-1, X.shape[-1]).T @ dpre.reshape(-1, dpre.shape[-1]); g["b1"] = dpre.sum((0, 1))
        dX = dpre @ self.P["W1"].T
        g["E"] = np.zeros_like(self.P["E"]); np.add.at(g["E"], ctx, dX[..., :self.K * self.d].reshape(N, T1, self.K, self.d))
        g["pos"] = np.zeros_like(self.P["pos"]); g["pos"][:T1] = dX[..., self.K * self.d:].sum(0)
        return g

    def adam_step(self, g, lr=3e-3, clip=1.0):
        norm = np.sqrt(sum((v ** 2).sum() for v in g.values())); scale = min(1.0, clip / (norm + 1e-12)); self.t += 1
        for k in self.P:
            gk = g[k] * scale; self.m[k] = 0.9 * self.m[k] + 0.1 * gk; self.v[k] = 0.999 * self.v[k] + 0.001 * gk ** 2
            self.P[k] -= lr * (self.m[k] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[k] / (1 - 0.999 ** self.t)) + 1e-8)

    def sample(self, n, seed=0, temperature=1.0):
        """Generate n sequences token by token."""
        rng = np.random.default_rng(seed); ids = np.full((n, MAXLEN), PAD); ids[:, 0] = BOS; done = np.zeros(n, bool)
        for t in range(1, MAXLEN):
            lsm = self.forward(ids[:, :t + 1])[:, t - 1] / temperature
            p = np.exp(lsm - lsm.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
            nxt = (p.cumsum(1) > rng.random((n, 1))).argmax(1)
            if t == MAXLEN - 1:
                nxt = np.full(n, EOS)                                                # every sequence ends by the cap
            nxt[done] = PAD; ids[:, t] = nxt; done |= nxt == EOS
            if done.all():
                break
        return ids

In [ ]:
import time
def train_sft(epochs=12, lr=3e-3, seed=RANDOM_STATE):
    """Supervised fine-tuning: maximise the log-probability of the corpus, token by token."""
    lm = NeuralLM(seed=seed); rng = np.random.default_rng(seed)
    for ep in range(epochs):
        for b in np.array_split(rng.permutation(len(IDS)), len(IDS) // 128):
            batch = IDS[b]; w = np.ones(len(b)) / ((batch[:, 1:] != PAD).sum())
            lm.adam_step(lm.grad_weighted(batch, w), lr=lr)
    return lm

t0 = time.time(); sft = train_sft()
lp, mask = sft.token_logprobs(IDS[:500])
print(f"SFT trained in {time.time() - t0:.1f} s; mean token log-probability on the corpus {(lp * mask).sum() / mask.sum():.3f}")
o_sft = sft.sample(512, seed=1); t_sft = [decode(r) for r in o_sft]
g_sft = np.array([gold_reward(t) for t in t_sft])
print(f"SFT policy, gold reward over 512 samples: mean {g_sft.mean():.3f}, standard deviation {g_sft.std():.3f}")
print(f"share of samples that follow the grammar of the corpus: {np.mean([t in GRAMMAR for t in t_sft]):.3f}")
for t in t_sft[:5]:
    print(f"   {gold_reward(t):+.2f}  {t}")

In [ ]:
def make_pairs(model, n_pairs=1500, noise=0.10, seed=7):
    """Sample two completions and ask a noisy annotator, who follows the gold reward, which one is better."""
    rng = np.random.default_rng(seed); o = model.sample(2 * n_pairs, seed=seed); txt = [decode(r) for r in o]
    W, L, flipped = [], [], 0
    for i in range(n_pairs):
        ga, gb = gold_reward(txt[2 * i]), gold_reward(txt[2 * i + 1])
        if abs(ga - gb) < 1e-9:
            continue                                        # a tie: an annotator would skip it
        win, lose = (o[2 * i], o[2 * i + 1]) if ga > gb else (o[2 * i + 1], o[2 * i])
        if rng.random() < noise:
            win, lose = lose, win; flipped += 1             # an annotation error
        W.append(win); L.append(lose)
    return np.array(W), np.array(L), flipped

Wp, Lp, flipped = make_pairs(sft, noise=0.10)
print(f"{len(Wp)} usable pairs, {flipped} labels flipped by annotator error ({flipped / len(Wp):.1%})")
gv = np.array([gold_reward(decode(r)) for r in sft.sample(1000, seed=11)]); diffs = np.abs(gv[0::2] - gv[1::2])
print(f"median gold difference within a pair: {np.median(diffs):.3f}; pairs closer than 0.2: {(diffs < 0.2).mean():.1%}")

In [ ]:
def adam(P, g, state, lr):
    state["t"] = state.get("t", 0) + 1; t = state["t"]
    for k in P:
        m = state.setdefault("m" + k, np.zeros_like(P[k])); v = state.setdefault("v" + k, np.zeros_like(P[k]))
        m[...] = 0.9 * m + 0.1 * g[k]; v[...] = 0.999 * v + 0.001 * g[k] ** 2
        P[k] -= lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)

class RewardModel:
    """The mean of the token embeddings and the length feed a small network that outputs one score.
    Like every learned reward, it is an approximation: It sees which words occur, not their order or repetition."""
    def __init__(self, d=16, H=32, seed=0):
        rng = np.random.default_rng(seed)
        self.P = {"E": rng.normal(0, 0.3, (V, d)), "U1": rng.normal(0, 1 / np.sqrt(d + 1), (d + 1, H)), "c1": np.zeros(H),
                  "u2": rng.normal(0, 1 / np.sqrt(H), H), "c2": np.zeros(1)}
        self.state = {}
    def score(self, ids):
        tok = ids > EOS; n = tok.sum(1, keepdims=True).clip(1)
        F = np.concatenate([(self.P["E"][ids] * tok[..., None]).sum(1) / n, n / 10.0], 1)
        Hh = np.tanh(F @ self.P["U1"] + self.P["c1"]); self.cache = (ids, F, tok, n, Hh)
        return Hh @ self.P["u2"] + self.P["c2"][0]
    def grad(self, gs):
        ids, F, tok, n, Hh = self.cache
        g = {"u2": Hh.T @ gs, "c2": np.array([gs.sum()])}; dpre = np.outer(gs, self.P["u2"]) * (1 - Hh ** 2)
        g["U1"] = F.T @ dpre; g["c1"] = dpre.sum(0); dF = dpre @ self.P["U1"].T
        g["E"] = np.zeros_like(self.P["E"]); np.add.at(g["E"], ids, dF[:, None, :-1] * tok[..., None] / n[..., None])
        return g

def train_rm(W, L, epochs=30, lr=3e-3, seed=0, holdout=0.15):
    """Bradley-Terry: maximise log sigmoid(r(winner) - r(loser)) over the training pairs."""
    rm = RewardModel(seed=seed); rng = np.random.default_rng(seed); ntr = int((1 - holdout) * len(W))
    for ep in range(epochs):
        for b in np.array_split(rng.permutation(ntr), max(1, ntr // 128)):
            sw = rm.score(W[b]); sl = rm.score(L[b]); q = 1 / (1 + np.exp(sw - sl))     # q = 1 - sigmoid(sw - sl)
            rm.score(W[b]); gw = rm.grad(-q / len(b)); rm.score(L[b]); gl = rm.grad(q / len(b))
            adam(rm.P, {k: gw[k] + gl[k] for k in gw}, rm.state, lr)
    acc = float((rm.score(W[ntr:]) > rm.score(L[ntr:])).mean())
    return rm, acc

rm, acc = train_rm(Wp, Lp)
rm_on_sft = rm.score(o_sft); rho = float(np.corrcoef(rm_on_sft, g_sft)[0, 1])
print(f"reward model: held-out pairwise accuracy {acc:.3f}; correlation with the gold reward on SFT samples {rho:.3f}")

In [ ]:
def rlhf(sft, rm, beta=0.2, iters=300, batch=64, lr=1e-3, seed=0, track=25, eval_n=192):
    """REINFORCE with the KL penalty folded into the reward:  r(y) = RM(y) - beta * [log pi(y) - log pi_ref(y)]."""
    pol, ref = sft.copy(), sft; hist = []
    for it in range(iters):
        seqs = pol.sample(batch, seed=1000 * seed + it)
        kl = pol.seq_logprob(seqs) - ref.seq_logprob(seqs)                # a one-sample estimate of the KL
        reward = rm.score(seqs) - beta * kl
        adv = (reward - reward.mean()) / (reward.std() + 1e-6)
        pol.adam_step(pol.grad_weighted(seqs, adv / batch), lr=lr)
        if it % track == 0 or it == iters - 1:
            ev = pol.sample(eval_n, seed=50000 + it); tt = [decode(r) for r in ev]
            hist.append(dict(iter=it, gold=np.mean([gold_reward(t) for t in tt]), reward_model=float(rm.score(ev).mean()),
                             kl=float((pol.seq_logprob(ev) - ref.seq_logprob(ev)).mean()),
                             length=np.mean([len(t.split()) for t in tt]), in_grammar=np.mean([t in GRAMMAR for t in tt])))
    return pol, pd.DataFrame(hist)

## 2. Direct preference optimisation &#91;[1](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-1)&#93;

*Lecture: [From the optimum to DPO](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#from-the-optimum-to-dpo)*

Direct preference optimisation trains the policy on the preference pairs themselves.

**Python move: the same backward pass with pair weights.** The winner of each pair gets the weight `beta * q / B` and the loser its negative, where `q` is the gradient weight of the lecture and `B` the number of pairs in the mini-batch.

**From the formulas to the code.** In `dpo`, `pol` is the policy $\pi_\theta$ and `ref` the reference $\pi_{\mathrm{ref}}$, whose log-probabilities `lw_ref` and `ll_ref` are computed once. The variable `z` of the code is the product $\beta z$ of the lecture, so `q = 1 / (1 + np.exp(z))` is the gradient weight $q = \sigma(-\beta z)$. The two calls of `grad_weighted`, with the weights `beta * q / len(b)` for the winners and their negatives for the losers, add up to the mean gradient of the DPO loss over the mini-batch. `adam_step` first shortens a gradient longer than 1 to the length 1 and then takes the step with the learning rate `lr`. The loss itself is never computed, because the step needs only its gradient. In `hist`, `kl` is the estimate of the KL divergence from 256 sentences that are sampled after each epoch for measuring only, and `pair_accuracy` is the share of positive margins in the last mini-batch of the epoch.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rlhf" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def dpo(sft, W, L, beta=0.1, epochs=8, lr=1e-3, seed=0, batch=64, eval_n=256):
    """Direct preference optimisation: maximise log sigmoid(beta * [(log pi - log pi_ref)(y_w) - (log pi - log pi_ref)(y_l)])."""
    pol, ref = sft.copy(), sft; rng = np.random.default_rng(seed)
    lw_ref, ll_ref = ref.seq_logprob(W), ref.seq_logprob(L); hist = []
    for ep in range(epochs):
        for b in np.array_split(rng.permutation(len(W)), max(1, len(W) // batch)):
            z = beta * ((pol.seq_logprob(W[b]) - lw_ref[b]) - (pol.seq_logprob(L[b]) - ll_ref[b]))
            q = 1 / (1 + np.exp(z))                                       # weight: large for pairs ranked wrongly
            gw = pol.grad_weighted(W[b], beta * q / len(b)); gl = pol.grad_weighted(L[b], -beta * q / len(b))
            pol.adam_step({k: gw[k] + gl[k] for k in gw}, lr=lr)
        ev = pol.sample(eval_n, seed=7000 + ep); tt = [decode(r) for r in ev]
        hist.append(dict(epoch=ep + 1, gold=np.mean([gold_reward(t) for t in tt]),
                         kl=float((pol.seq_logprob(ev) - ref.seq_logprob(ev)).mean()),
                         pair_accuracy=float((z > 0).mean()), in_grammar=np.mean([t in GRAMMAR for t in tt])))
    return pol, pd.DataFrame(hist)

t0 = time.time(); dpo_pol, dh = dpo(sft, Wp, Lp, beta=0.1); T_DPO = time.time() - t0
print(f"DPO finished in {T_DPO:.1f} s\n"); print(dh.round(3).to_string(index=False))

**Python note.** `{k: gw[k] + gl[k] for k in gw}` is a dictionary comprehension: It builds a new dictionary with the same keys as `gw`, each holding the sum of the gradient for the winners and the gradient for the losers.

**Observe and reflect.** In the printed table the pair accuracy stays below 0.9 while the gold reward rises. Why can a policy that follows the gold reward never agree with all the labels of the pairs of Day 4?

## 3. DPO against RLHF on matched data

*Lecture: [DPO against RLHF on matched data](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#dpo-against-rlhf-on-matched-data)*

**From the formulas to the code.** The first line runs `rlhf` of section 1 with $\beta = 0.2$ for 300 iterations. `summary` draws 512 sentences with `model.sample(n, seed=321)` and computes the columns of the table of the lecture. `gold` is the mean gold reward, `kl` the mean of $\log \pi_\theta(y) - \log \pi_{\mathrm{ref}}(y)$ with `sft` as the reference, `length` the mean number of words, `in_grammar` the grammaticality and `distinct` the number of different sentences. The loop at the end repeats `dpo` for four other values of $\beta$ and prints the last row of each history, which is measured on 256 sentences. The column `seconds` is the measured training time and changes from run to run.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "T_DPO" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
t0 = time.time(); rlhf_pol, rh = rlhf(sft, rm, beta=0.2, iters=300, seed=0); T_RLHF = time.time() - t0
def summary(model, name, secs, n=512):
    ev = model.sample(n, seed=321); tt = [decode(r) for r in ev]
    return dict(method=name, gold=np.mean([gold_reward(t) for t in tt]), kl=float((model.seq_logprob(ev) - sft.seq_logprob(ev)).mean()),
                length=np.mean([len(t.split()) for t in tt]), in_grammar=np.mean([t in GRAMMAR for t in tt]),
                distinct=len(set(tt)), seconds=secs)
comp = pd.DataFrame([summary(sft, "SFT, no alignment", 0.0), summary(rlhf_pol, "RLHF, beta 0.2", T_RLHF),
                     summary(dpo_pol, "DPO, beta 0.1", T_DPO)])
print(comp.round(3).to_string(index=False))
print("\nDPO for other values of beta:")
for beta in (0.02, 0.05, 0.3, 1.0):
    _, h_ = dpo(sft, Wp, Lp, beta=beta, epochs=8, seed=0)
    print(f"   beta {beta:<5} gold {h_.gold.iloc[-1]:.3f}   KL {h_.kl.iloc[-1]:6.2f}   in grammar {h_.in_grammar.iloc[-1]:.2f}")

In [ ]:
kl_dpo = float(comp.loc[comp.method == "DPO, beta 0.1", "kl"].iloc[0])
kl_rlhf = float(comp.loc[comp.method == "RLHF, beta 0.2", "kl"].iloc[0])
print(f"KL divergence from the fine-tuned model: DPO {kl_dpo:.2f}, RLHF {kl_rlhf:.2f}")

**Observe and reflect.** DPO ends further from the fine-tuned model than RLHF and writes more sentences outside the grammar. Which property of the DPO loss lets the probability of sentences that occur in no pair change freely?

## 4. Three judges &#91;[2](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-2), [3](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-3)&#93;

*Lecture: [Judges and their biases](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#judges-and-their-biases)*

**From the formulas to the code.** `judge_from(score)` returns the score $s$ of one comparison: 1, 0.5 or 0. In `win_rate`, the array `s` holds the $n$ scores, `s.mean()` is the win rate $\hat{p}$ with ties counted as half, and `(s == 0.5).mean()` is the share of ties. The line with `choice(s, (n_boot, n)).mean(1)` draws the 2000 resamples of the bootstrap and computes their means, and `np.percentile` returns the 2.5th and the 97.5th percentile. Model A is sampled with `seed` and model B with `seed + 1`. `decided` is true when one half lies outside the interval, and `strict` is the win rate with ties counted as losses.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "kl_rlhf" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def win_rate(model_a, model_b, judge, n=400, seed=13, n_boot=2000):
    """Mean judgement of A against B (win 1, tie 0.5, loss 0) with a 95 percent bootstrap interval."""
    ta = [decode(r) for r in model_a.sample(n, seed=seed)]; tb = [decode(r) for r in model_b.sample(n, seed=seed + 1)]
    s = np.array([judge(x, y) for x, y in zip(ta, tb)], float)
    boot = np.random.default_rng(seed).choice(s, (n_boot, n)).mean(1)
    return s.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5), (s == 0.5).mean()

def judge_from(score):
    """A judge that compares two outputs by a score: 1 if the first is higher, 0.5 for a tie, 0 otherwise."""
    return lambda x, y: 1.0 if score(x) > score(y) else (0.5 if score(x) == score(y) else 0.0)

gold_judge = judge_from(gold_reward)
rm_judge = judge_from(lambda t: float(rm.score(np.array([encode(t)]))[0]))
length_judge = judge_from(lambda t: len(t.split()))                  # a judge that counts words
rows = []
for nm, m in (("RLHF", rlhf_pol), ("DPO", dpo_pol)):
    for jn, j in (("gold judge", gold_judge), ("reward-model judge", rm_judge), ("length judge", length_judge)):
        p, lo, hi, ties = win_rate(m, sft, j)
        rows.append(dict(model=nm, judge=jn, win_rate_vs_SFT=p, low=lo, high=hi, ties=ties, decided=not (lo <= 0.5 <= hi)))
wr = pd.DataFrame(rows)
print(wr.round(3).to_string(index=False))
strict = np.mean([gold_reward(x) > gold_reward(y) for x, y in zip([decode(r) for r in rlhf_pol.sample(400, seed=13)],
                                                                  [decode(r) for r in dpo_pol.sample(400, seed=14)])])
print(f"\nRLHF against DPO under the gold judge: {win_rate(rlhf_pol, dpo_pol, gold_judge)[0]:.3f} with ties counted as half, "
      f"{strict:.3f} if ties count as losses")

**Python note.** `judge_from(score)` returns a function, written with `lambda`, that compares two sentences by `score`. A function that builds other functions keeps the code short: Each of the three judges takes one line.

**Observe and reflect.** Under which judge does DPO look best, and under which worst? If you could report only one judge, which would you choose, and what would you check before trusting it?

**Exercise A.** Store the win rate of DPO against SFT under the length judge in `wr_len`.

In [ ]:
wr_len = None   # your answer

check("Exercise A", wr_len, float(wr[(wr.model == "DPO") & (wr.judge == "length judge")].win_rate_vs_SFT.iloc[0]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
wr_len = float(wr[(wr.model == "DPO") & (wr.judge == "length judge")].win_rate_vs_SFT.iloc[0])

check("Exercise A", wr_len, float(wr[(wr.model == "DPO") & (wr.judge == "length judge")].win_rate_vs_SFT.iloc[0]))

## 5. Intervals, sample size and seeds &#91;[4](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-4)&#93;

*Lecture: [Intervals, sample size and seeds](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#intervals-sample-size-and-seeds)*

**From the formulas to the code.** The loop calls `win_rate` with $n$ = 25, 100, 400 and 1600 and prints the bootstrap interval of each comparison. `rlhf_seed1` is trained by the same call as `rlhf_pol` with `seed=1`, which changes only the sentences that are sampled during training. Exercise C computes the half-width $1.96 \sqrt{p(1 - p) / n}$ of the lecture at $p = 0.5$, where $p(1 - p) = 0.25$. The bootstrap intervals of the two seeds are narrower than this formula says, because more than 40 percent of their comparisons are ties. To see whether a decision survives fresh sentences, call `win_rate` again with another value of `seed`, for example `win_rate(rlhf_pol, rlhf_seed1, gold_judge, n=1600, seed=13)`.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "wr_len" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
rlhf_seed1, _ = rlhf(sft, rm, beta=0.2, iters=300, seed=1)          # the same recipe, another seed
for label, a, b in (("RLHF against SFT: a large difference", rlhf_pol, sft),
                    ("RLHF against RLHF with another seed: no designed difference", rlhf_pol, rlhf_seed1)):
    print(label)
    for n in (25, 100, 400, 1600):
        p, lo, hi, _ = win_rate(a, b, gold_judge, n=n, seed=99)
        print(f"   n = {n:4d}   win rate {p:.3f}   interval [{lo:.3f}, {hi:.3f}]   decided: {not (lo <= 0.5 <= hi)}")

**Observe and reflect.** With 1600 comparisons the two RLHF runs, which differ only in their seed, may count as decided. Run the loop again with `seed=7` in place of `seed=99` in the call of `win_rate`. Does the decision survive fresh sentences, and what does that say about a single interval?

**Exercise B.** Compute the half-width of a 95 percent interval for a win rate near one half with 400 comparisons, `1.96 * np.sqrt(0.25 / 400)`, and store it in `hw400`.

In [ ]:
hw400 = None   # your answer

check("Exercise B", hw400, 0.049)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
hw400 = 1.96 * np.sqrt(0.25 / 400)

check("Exercise B", hw400, 0.049)

## 6. Evaluating safety: two failure rates with intervals &#91;[14](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-14)&#93;

*Lecture: [Evaluating safety](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#evaluating-safety)*

A safety test set holds harmful requests and harmless requests that resemble them. A judge marks every harmful completion and every refusal of a harmless request, and each failure rate is reported with its interval. The counts below are the invented counts of the worked example of the lecture.

**From the formulas to the code.** `rate_interval(k, n)` builds a list of `n` zeros and ones with `k` ones, one per request of the test set, and returns the failure rate $\hat{p} = k / n$ with the 2.5th and the 97.5th percentile of 2000 bootstrap means, the interval of the lecture without the normal approximation. For `k = 0` every resample is zero and the interval collapses to a point, so the last line prints the rule of three, $3 / n$, instead.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "hw400" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

**Python note.** `np.r_[np.ones(k), np.zeros(n - k)]` joins two arrays into one, here `k` ones followed by `n - k` zeros. `rng.choice(x, (n_boot, n))` draws a table of `n_boot` rows of `n` elements of `x` with replacement, so `.mean(1)`, the mean of every row, gives all bootstrap means in one line.

In [ ]:
def rate_interval(k, n, n_boot=2000, seed=0):
    """Failure rate k / n of a test set and its 95 percent bootstrap interval."""
    x = np.r_[np.ones(k), np.zeros(n - k)]
    boot = np.random.default_rng(seed).choice(x, (n_boot, n)).mean(1)
    return k / n, np.percentile(boot, 2.5), np.percentile(boot, 97.5)

TEST = {"harmful requests completed": (200, {"model A": 6, "model B": 1}),
        "harmless requests refused": (250, {"model A": 4, "model B": 21})}
for failure, (n, counts) in TEST.items():
    print(f"{failure}, out of {n}:")
    for model, k in counts.items():
        p, lo, hi = rate_interval(k, n)
        print(f"   {model}: {k:3d}   rate {p:.3f}   interval {lo:.3f} to {hi:.3f}")

print("\nNo failure at all: upper end of the interval by the rule of three, 3 / n")
for n in (50, 100, 200, 500, 1000):
    print(f"   n = {n:4d}: {3 / n:.3f}")

**Observe and reflect.** Model B is safer on harmful requests and refuses more harmless ones. Which model would you deploy for a homework helper for children, and which for a medical information service? Roughly how many harmful requests would a test set need before a rate of 0.005 could be told apart from one of 0.03?

## 7. Your judge

*Lecture: [Your judge](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#your-judge)*

Change `JUDGE` to `short`, `no repetition` or `good opening` and run the cell again.

**From the formulas to the code.** `MY_SCORES` holds the four scoring functions of the table of the lecture. `CALM.intersection(t.split())` is the set of the different calm words of a sentence, `len(t.split())` its number of words and `len(set(t.split()))` its number of different words. The minus signs turn *fewer is better* into a higher score, because `judge_from` always prefers the higher score. `my_table` and the bar chart show the win rate of each model against the fine-tuned model with its bootstrap interval.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "TEST" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
JUDGE = "calm words"     # choose: calm words, short, no repetition or good opening

MY_SCORES = {
    "calm words": lambda t: len(CALM.intersection(t.split())),
    "short": lambda t: -len(t.split()),
    "no repetition": lambda t: -(len(t.split()) - len(set(t.split()))),
    "good opening": lambda t: float(t.split()[0] in ("the", "a", "my")) if t.split() else 0.0,
}
my_judge = judge_from(MY_SCORES[JUDGE]); judge_rows = []
for nm, m in (("RLHF", rlhf_pol), ("DPO", dpo_pol)):
    p, lo, hi, ties = win_rate(m, sft, my_judge)
    judge_rows.append(dict(model=nm, judge=JUDGE, win_rate_vs_SFT=p, low=lo, high=hi, ties=ties, decided=not (lo <= 0.5 <= hi)))
my_table = pd.DataFrame(judge_rows); print(my_table.round(3).to_string(index=False))
ax = my_table.set_index("model").win_rate_vs_SFT.plot.barh(figsize=(6, 2.4), color="#2980b9",
     xerr=[my_table.win_rate_vs_SFT - my_table.low, my_table.high - my_table.win_rate_vs_SFT])
ax.axvline(0.5, color="k", lw=.8); ax.set_xlim(0, 1); ax.set_title(f"win rate against SFT, judge: {JUDGE}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
dpo_decided = bool(my_table.set_index("model").loc["DPO", "decided"])
print(JUDGE, "-> comparison of DPO against SFT decided:", dpo_decided)

**Observe and reflect.** Change `JUDGE` once more and compare. Which judges reward a property that the gold reward also rewards, and which reward a surface property? Write a judge of your own into `MY_SCORES` that you would trust more.

## 8. The final project

*Lecture: [Open problems and the final project](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#open-problems-and-the-final-project)*

The brief is in the folder `exams/final` of the repository: An alignment experiment with a gold reward the pipeline never sees, two methods on matched data and compute, an overoptimisation study and an evaluation with intervals, a length control and an independent judge &#91;[17](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lecture.html#ref-17)&#93;.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "dpo_decided" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-05/NB05_dpo_and_evaluation.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-8":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Implement the loss of identity preference optimisation (IPO), given in the lecture, in place of the DPO loss and compare their drift for small beta on the data of the notebook. |
| Electronics and Communication Engineering | Add position bias to the judge, preferring the first output in 20 percent of close calls, and show how swapping the order removes it. |
| Electrical and Electronics Engineering | Plan an evaluation that detects a win rate of 0.53 with 95 percent confidence and state the number of comparisons and seeds it needs. |
| Mechanical Engineering | Train DPO for 2, 8 and 32 epochs and plot gold reward, KL and grammaticality against the number of epochs. |
| Biomedical Engineering | Design a gold reward and a length-controlled judge for short patient instructions and explain why the judge must not prefer longer outputs. |
| Civil Engineering | Compare RLHF and DPO over five seeds each and report the difference with an interval over seeds. |
| Aeronautical Engineering | Train the reward model on half of the pairs and DPO on the same half, and compare how the two methods degrade. |
| Biotechnology | Replace the gold judge by a judge that sees only the calm words and report which conclusions of section 4 change. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-05/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-05).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>